# explanation:

here we go through CS and UIC patients and get the LTM reports pdf, and copy and paste them to a folder here

This script:
1. Reads Patient_ID values from the CSV.
2. Creates the destination folder.
 3. Searches Z:\Data for folders beginning with CS or UIC that contain a Patient_ID.
4. Searches those folders recursively for PDFs containing "LTMreport"
 in the filename (case-insensitive).
 5. Copies each report and renames it as Patient_ID.pdf.

In [ ]:
import os
import re
import shutil
from pathlib import Path
import pandas as pd

In [ ]:
csv_path = Path(
    r"D:\Nill\code\BART\IED\0_0_new_IED_new_area_labels\0_demographic"
) / "BART_Subject_demographics_SOZ.csv"

source_root = Path(r"Z:\Data")

destination_folder = Path(
    r"D:\Nill\code\BART\IED\0_0_new_IED_new_area_labels"
) / "10_1_SOZ_processing_collecting_LTM_reports"


if not csv_path.is_file():
    raise FileNotFoundError(
        f"The CSV file was not found:\n{csv_path}"
    )

if not source_root.is_dir():
    raise FileNotFoundError(
        f"The source directory is unavailable:\n{source_root}\n\n"
        "Open the Z: drive in File Explorer and make sure it is connected."
    )


destination_folder.mkdir(parents=True, exist_ok=True)

print(f"Destination folder:\n{destination_folder}\n")



df = pd.read_csv(csv_path, dtype={"Patient_ID": str})

if "Patient_ID" not in df.columns:
    raise KeyError(
        f"The CSV does not contain a Patient_ID column:\n{csv_path}"
    )

patient_ids = (
    df["Patient_ID"]
    .dropna()
    .str.strip()
    .loc[lambda values: values != ""]
    .drop_duplicates()
    .tolist()
)

print("Patient IDs:")
print(patient_ids)
print("Number of unique Patient IDs:", len(patient_ids))
print()


In [ ]:
def find_patient_id_in_folder(folder_name, ids):
    for patient_id in ids:
        pattern = rf"(?<!\d){re.escape(patient_id)}(?!\d)"

        if re.search(pattern, folder_name, flags=re.IGNORECASE):
            return patient_id

    return None


def find_ltm_reports_safely(patient_folder):
    reports = []

    def handle_walk_error(error):
        print(f"  Warning: inaccessible folder was skipped: {error}")

    for folder_path, _, filenames in os.walk(
        patient_folder,
        topdown=True,
        onerror=handle_walk_error,
        followlinks=False,
    ):
        for filename in filenames:
            filename_lower = filename.lower()

            if (
                filename_lower.endswith(".pdf")
                and "ltmreport" in filename_lower
            ):
                reports.append(Path(folder_path) / filename)

    return sorted(
        reports,
        key=lambda path: str(path).lower(),
    )




In [ ]:
copied_patient_ids = set()
matching_folders = []
multiple_reports = []
missing_reports = []
copy_errors = []
source_walk_errors = []


def handle_source_walk_error(error):
    source_walk_errors.append(str(error))
    print(f"Warning: inaccessible source folder was skipped: {error}")


print("Searching for CS and UIC patient folders...\n")

for current_root, directory_names, _ in os.walk(
    source_root,
    topdown=True,
    onerror=handle_source_walk_error,
    followlinks=False,
):
    current_root = Path(current_root)

    # Work on a copy because os.walk may update directory_names.
    for directory_name in list(directory_names):

        # Folder name must start with CS or UIC,
        # ignoring capitalization.
        if not directory_name.upper().startswith(("CS", "UIC")):
            continue

        patient_id = find_patient_id_in_folder(
            directory_name,
            patient_ids,
        )

        if patient_id is None:
            continue

        patient_folder = current_root / directory_name
        matching_folders.append((patient_id, patient_folder))

        print(f"Matched Patient_ID {patient_id}:")
        print(f"  Folder: {patient_folder}")

        # Search recursively for LTMreport PDFs while safely skipping
        # inaccessible files and folders.
        ltm_reports = find_ltm_reports_safely(patient_folder)

        if not ltm_reports:
            print("  No LTMreport PDF was found.\n")
            missing_reports.append((patient_id, patient_folder))
            continue

        if len(ltm_reports) > 1:
            print(
                f"  Warning: {len(ltm_reports)} "
                "LTMreport PDFs were found."
            )
            print("  The first PDF will be copied:")

            for report in ltm_reports:
                print(f"    {report}")

            multiple_reports.append(
                (patient_id, patient_folder, ltm_reports)
            )

        selected_report = ltm_reports[0]
        destination_pdf = destination_folder / f"{patient_id}.pdf"

        # Avoid copying a second report if this Patient_ID matched
        # another folder during the current run.
        if patient_id in copied_patient_ids:
            print(
                f"  Skipped: a report for Patient_ID {patient_id} "
                "has already been copied.\n"
            )
            continue

        try:
            # copy2 copies the file and preserves its metadata.
            shutil.copy2(selected_report, destination_pdf)

            copied_patient_ids.add(patient_id)

            print(f"  Source PDF: {selected_report}")
            print(f"  Copied as:  {destination_pdf.name}\n")

        except OSError as error:
            copy_errors.append(
                (patient_id, selected_report, str(error))
            )
            print(f"  Copy error: {error}\n")





In [ ]:
patient_ids_not_copied = sorted(
    set(patient_ids) - copied_patient_ids
)


missing_reports_csv = (
    destination_folder / "patients_without_LTM_report.csv"
)

missing_df = pd.DataFrame({
    "Patient_ID": patient_ids_not_copied
})

missing_df.to_csv(missing_reports_csv, index=False)


# ---------------------------------------------------------------------
# Print summary
# ---------------------------------------------------------------------

print("=" * 70)
print("SUMMARY")
print("=" * 70)

print(f"Total Patient_ID values:       {len(patient_ids)}")
print(f"Matching patient folders:      {len(matching_folders)}")
print(f"Reports successfully copied:   {len(copied_patient_ids)}")
print(f"Patients without copied PDFs:  {len(patient_ids_not_copied)}")
print(f"Folders with multiple reports: {len(multiple_reports)}")
print(f"Copy errors:                   {len(copy_errors)}")
print(f"Source folders skipped:        {len(source_walk_errors)}")

if copied_patient_ids:
    print("\nSuccessfully copied Patient_IDs:")

    for patient_id in sorted(copied_patient_ids):
        print(f"  {patient_id}.pdf")

if patient_ids_not_copied:
    print("\nPatient_IDs for which no report was copied:")

    for patient_id in patient_ids_not_copied:
        print(f"  {patient_id}")

if multiple_reports:
    print("\nFolders containing multiple LTM reports:")

    for patient_id, patient_folder, reports in multiple_reports:
        print(f"  Patient_ID: {patient_id}")
        print(f"  Folder: {patient_folder}")

        for report in reports:
            print(f"    {report}")

if copy_errors:
    print("\nCopy errors:")

    for patient_id, report_path, error_message in copy_errors:
        print(f"  Patient_ID: {patient_id}")
        print(f"  PDF: {report_path}")
        print(f"  Error: {error_message}")

if source_walk_errors:
    print("\nInaccessible source folders that were skipped:")

    for error_message in source_walk_errors:
        print(f"  {error_message}")

print(f"\nReports were saved in:\n{destination_folder}")

print(
    "\nPatients without a copied LTM report were saved to:"
    f"\n{missing_reports_csv}"
)